IMPORT ALL THE LIBRARIES

## 🔐 Security & Privacy

This public version intentionally does **not** contain:

- Google Cloud service-account credentials
- Google Drive folder IDs
- API keys or passwords
- Downloaded financial/customer data
- Customer-level records or personally identifiable information

The notebook reads credentials and the Drive folder ID from local environment
variables. Keep the real credential JSON file outside the Git repository.

For a public portfolio repository, clear notebook outputs before committing
and use only synthetic or anonymized sample data for demonstrations.


In [ ]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
import pandas as pd
import requests
from io import StringIO
from io import BytesIO  # For handling Excel files

## Google Drive Authentication Configuration

The credentials are loaded from an environment variable so that
service-account secrets are never stored in this notebook.

Set `GOOGLE_SERVICE_ACCOUNT_FILE` locally to the path of your
Google Cloud service-account JSON file.


In [ ]:
import os

# Local-only configuration.
# Do NOT place the service-account JSON file in this repository.
SERVICE_ACCOUNT_FILE = os.getenv("GOOGLE_SERVICE_ACCOUNT_FILE")

if not SERVICE_ACCOUNT_FILE:
    raise EnvironmentError(
        "Set the GOOGLE_SERVICE_ACCOUNT_FILE environment variable "
        "to your local service-account JSON file."
    )

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

# The Drive folder ID is also kept outside the public notebook.
FOLDER_ID = os.getenv("GOOGLE_DRIVE_FOLDER_ID")

if not FOLDER_ID:
    raise EnvironmentError(
        "Set the GOOGLE_DRIVE_FOLDER_ID environment variable "
        "to the ID of the Drive folder used for the project."
    )


AUTHETICATE AND CREATE THE SERVICES

In [ ]:
# Authenticate and create the Google Drive API service

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=SCOPES
)

service = build("drive", "v3", credentials=credentials)


FOLDER ID

In [ ]:
# FOLDER_ID is loaded from the GOOGLE_DRIVE_FOLDER_ID
# environment variable in the configuration cell above.
FOLDER_ID


FUNCTION TO LIST ALL THE FILES IN GOOGLE DRIVE

In [ ]:
def list_files(service, folder_id):
    results = service.files().list(
        q=f"'{folder_id}' in parents",
        fields="files(id, name, mimeType)"
    ).execute()
    return results.get('files', [])


In [ ]:
# Fetch files from the configured Google Drive folder.
files = list_files(service, FOLDER_ID)

print(f"Files found in the configured Drive folder: {len(files)}")


In [ ]:
# Download supported files and store their contents as Pandas DataFrames.
file_dataframes = []

for file in files:
    file_id = file["id"]
    file_name = file["name"]
    mime_type = file["mimeType"]

    if mime_type == "application/vnd.google-apps.spreadsheet":
        download_url = (
            f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv"
        )
    elif mime_type == "text/csv":
        download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    elif mime_type == "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet":
        download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    else:
        continue

    response = requests.get(download_url)

    if response.status_code == 200:
        if mime_type == "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet":
            df = pd.read_excel(BytesIO(response.content))
        else:
            df = pd.read_csv(StringIO(response.content.decode("utf-8")))

        file_dataframes.append(df)
    else:
        print("A file could not be downloaded successfully.")


In [ ]:
# Number of successfully loaded datasets
print(f"Datasets loaded successfully: {len(file_dataframes)}")


In [ ]:
# Combine all dataframes into a single one
if file_dataframes:
    combined_df = pd.concat(file_dataframes, ignore_index=True)

In [ ]:
# Validate the consolidated dataset without displaying sensitive records.
if "combined_df" in globals():
    print(f"Combined dataset shape: {combined_df.shape}")
else:
    print("No combined dataset was created.")
